# 03. The fair ranking

A carrier's average delay mixes how well it runs with where and when it flies. The fair ranking
regresses arrival delay on carrier with fixed effects for route, month, scheduled hour and aircraft
type, fit on weighted cells. The simulator plants carrier effects on routes of known difficulty
and makes the hardest routes belong to some carriers more than others, so the raw ranking is
confounded by design and the true ranking is known.

The dead end is kept: a model with fixed effects for the origin airport instead of the route. It
puts each carrier's hub in a fixed effect that the carrier owns, and the hub soaks up the carrier.

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)
ROOT = Path(os.environ.get("TURNAROUND_NOTEBOOK_ROOT") or (Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent))
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values, tables = manifest["values"], manifest["tables"]


def v(key):
    return values[key]["value"]


def t(key):
    # A manifest value formatted the way the documents print it.
    from turnaround_core.formats import format_value

    return format_value(values[key]["value"], values[key]["fmt"])


def table(key):
    entry = tables[key]
    return pl.DataFrame(entry["rows"], schema=entry["columns"], orient="row")


MARTS = ROOT / "results" / "marts"
print("manifest as of", manifest["as_of"], "with", len(values), "values;", "window", v("data.first_month"), "to", v("data.last_month"))

manifest as of 2026-10-01 with 625 values; window January 2015 to July 2026


In [2]:
from turnaround_chapters.ranking import estimate, spearman
from turnaround_sim.network import SimSpec, simulate
from turnaround_stats.fe import contrasts_to_mean, dummies, fit


def ranked(con, keys):
    """Carriers ordered by their effect under the fixed effects named in keys."""
    cells = con.execute(
        f"select carrier, {', '.join(keys)}, count(*) as flights, avg(arr_delay) as arr_delay from f "
        "where not cancelled and not diverted and arr_delay is not null group by all order by all"
    ).pl()
    carriers = sorted(cells["carrier"].unique().to_list())
    index = {c: i for i, c in enumerate(carriers)}
    idx = np.array([index[c] for c in cells["carrier"].to_list()])
    w = cells["flights"].to_numpy().astype(float)
    result = fit(cells["arr_delay"].to_numpy().astype(float), dummies(idx, len(carriers)), w, [cells[k].to_numpy() for k in keys], carriers[1:])
    effects, _ = contrasts_to_mean(result.coef, None, np.bincount(idx, weights=w))
    return [c for _, c in sorted(zip(effects, carriers))]


rows = []
for seed in (1, 2, 3):
    sim = simulate(SimSpec(seed=seed, days=365, start="2022-01-01", confounding=0.9))
    con = duckdb.connect()
    con.register("sim", sim.flights.to_arrow())
    con.execute("create view f as select *, year * 100 + month as period from sim")
    truth = sim.truth.carrier_effect_ranking
    final = estimate(con, "f")
    rows.append(
        {
            "seed": seed,
            "raw": spearman(final.order("raw"), truth),
            "origin airport fixed effect (dead end)": spearman(ranked(con, ["origin", "period", "dep_hour"]), truth),
            "both airports": spearman(ranked(con, ["origin", "dest", "period", "dep_hour"]), truth),
            "route, month, hour, type (shipped)": spearman(final.order("adjusted"), truth),
        }
    )
print("rank correlation with the true ranking, strong confounding")
pl.DataFrame(rows)

rank correlation with the true ranking, strong confounding


seed,raw,origin airport fixed effect (dead end),both airports,"route, month, hour, type (shipped)"
i64,f64,f64,f64,f64
1,-0.286,0.214,0.952,0.976
2,-0.214,0.167,0.976,0.976
3,-0.381,-0.071,0.952,0.952


The origin airport model is barely better than the raw ranking. Every carrier in the simulator
has a hub, and nearly every departure from that hub is the carrier's own, so the hub's fixed
effect and the carrier's effect are the same variable on half the carrier's flights: the model
credits the hub with the carrier's punctuality and the carrier with whatever is left. Holding both
ends fixed recovers most of the ranking; the route, which holds the pair and its difficulty,
recovers it best and is what ships. The recovery study runs this across every condition; its
summary is in the manifest.

In [3]:
print("rank correlation, adjusted against raw, over the recovery study:",
      "strong confounding", t("recovery.rank_adjusted.strong"), "against", t("recovery.rank_raw.strong"), ";",
      "no confounding", t("recovery.rank_adjusted.none"), "against", t("recovery.rank_raw.none"))

rank correlation, adjusted against raw, over the recovery study: strong confounding 0.94 against -0.07 ; no confounding 0.97 against 0.75


## The ranking on the real flights

Raw beside adjusted, with day clustered intervals, from the committed manifest. A carrier that moves
down once adjusted was flying easier routes and hours than its raw rank credited.

In [4]:
table("ch5.ranking")

Carrier,Flights,Raw rank,Raw effect,Adjusted rank,Adjusted effect,Interval,Moved
str,i64,i64,f64,i64,f64,str,str
"""Endeavor Air""",391977,1,-7.385,1,-13.136,"""-14.14 to -12.14 min""","""same"""
"""Republic Airways""",1119192,2,-6.262,2,-8.697,"""-9.36 to -8.03 min""","""same"""
"""Envoy Air""",975162,5,-2.360,3,-6.101,"""-6.78 to -5.42 min""","""up 2"""
"""Delta Air Lines""",3572228,4,-3.625,4,-2.918,"""-3.26 to -2.58 min""","""same"""
"""United Airlines""",2729655,7,-1.176,5,-2.553,"""-2.90 to -2.21 min""","""up 2"""
"""SkyWest Airlines""",2713958,9,0.161,6,-2.382,"""-3.00 to -1.76 min""","""up 3"""
"""PSA Airlines""",784660,11,4.190,7,-1.576,"""-2.44 to -0.71 min""","""up 4"""
"""Southwest Airlines""",5003261,6,-1.631,8,1.709,"""+1.40 to +2.02 min""","""down 2"""
"""Alaska Airlines""",918478,3,-3.979,9,2.777,"""+2.39 to +3.16 min""","""down 6"""
